## Analisis inicial y Union
Primera parte del exploratorio, lo que quiero ver es como unir los 3 excels juntos y cuan mal están:

In [1]:
# Imports iniciales
from datetime import datetime, time, timedelta
from pathlib import Path
import pandas as pd

Los leo:

In [2]:

# Lo pongo así para que no se rompa el path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

EXCEL_DIR = ROOT / "data" / "excels"
archivos = sorted(EXCEL_DIR.glob("Totalizadores Planta *.xlsx"))

# Por ahora miro solo EE para no tener que ver muchas hojas a la vez:
HOJA_REF = "Consolidado EE"
LIMITE = 23 * 3600            # 23:00:00 en segundos desde medianoche

# Se leen de nuevo en cada ejecución
dfs = {}

for archivo in archivos:
    print("Leyendo", archivo.name)
    dfs[archivo.name] = pd.read_excel(archivo, sheet_name=HOJA_REF, engine="openpyxl")

for nombre, df in dfs.items():
    print(nombre, df.shape, list(df.columns[:3]))

Leyendo Totalizadores Planta 2020_2022.xlsx
Leyendo Totalizadores Planta 2021_2023.xlsx
Leyendo Totalizadores Planta 2022_2023.xlsx
Totalizadores Planta 2020_2022.xlsx (15575, 21) ['DIA', 'HORA', 'Planta (Kw)']
Totalizadores Planta 2021_2023.xlsx (15316, 21) ['DIA', 'HORA', 'Planta (Kw)']
Totalizadores Planta 2022_2023.xlsx (12009, 21) ['DIA', 'HORA', 'Planta (Kw)']


In [3]:
def a_segundos(v):
    """Convierte una celda de hora a segundos desde las 00:00, o None si no se entiende."""
    if v is None or (not isinstance(v, str) and pd.isna(v)):
        return None
    if isinstance(v, datetime):
        v = v.time()
    if isinstance(v, time):
        return v.hour * 3600 + v.minute * 60 + v.second
    if isinstance(v, timedelta):
        return int(v.total_seconds())
    if isinstance(v, (int, float)):
        return round(v * 86400)
    if isinstance(v, str):
        partes = v.strip().split(":")
        if len(partes) == 3 and all(p.isdigit() for p in partes):
            h, m, s = map(int, partes)
            return h * 3600 + m * 60 + s
    return None                               # formato raro, como "0:1:00:01"


def formato(seg):
    return f"{seg // 3600:02d}:{seg % 3600 // 60:02d}:{seg % 60:02d}"


def revisar_horas(df):
    """Horas posteriores a las 23:00:00 y valores de hora con formato raro."""
    horas = df.iloc[:, 1]                 # la segunda columna es la hora
    segundos = horas.map(a_segundos)

    tarde = segundos[segundos > LIMITE]
    print("Horas > 23:00:00 (valor: cantidad de filas)")
    for seg, n in tarde.value_counts().sort_index().items():
        print(f"  {formato(int(seg))}: {n}")

    raros = horas[segundos.isna() & horas.notna()]
    if len(raros):
        print("Valores con formato no reconocido:")
        for valor, n in raros.astype(str).value_counts().items():
            print(f"  {valor!r}: {n}")


def revisar_dias(df, min_filas=24, max_filas=25):
    """Filas por día, y un df aparte con los días con menos de min_filas o más de max_filas."""
    col_dia = df.columns[0]
    conteo = df.groupby(col_dia).size()

    print("Cantidad de días según filas por día:")
    print(conteo.value_counts().sort_index())

    dias_raros = conteo[(conteo < min_filas) | (conteo > max_filas)].index
    df_raros = df[df[col_dia].isin(dias_raros)].copy()
    return conteo, df_raros

Me ponga a ver como es el formato de las :59 

**queda justificar por que las :59**

In [4]:
for nombre, df in dfs.items():
    print(f"\n=== {nombre} ===")
    revisar_horas(df)


=== Totalizadores Planta 2020_2022.xlsx ===
Horas > 23:00:00 (valor: cantidad de filas)
  23:00:01: 10
  23:59:00: 614
  23:59:01: 7
  23:59:59: 1

=== Totalizadores Planta 2021_2023.xlsx ===
Horas > 23:00:00 (valor: cantidad de filas)
  23:00:01: 9
  23:59:00: 605
  23:59:01: 5
  23:59:59: 1

=== Totalizadores Planta 2022_2023.xlsx ===
Horas > 23:00:00 (valor: cantidad de filas)
  23:00:01: 5
  23:59:00: 475
  23:59:01: 3
  23:59:59: 1


La mayoría de horas están bien, hay que tener en cuenta las 23:00:01 seguramente son duplicados de 23:00 y no de :59. 

In [5]:
conteos = {}
lista_raros = []

for nombre, df in dfs.items():
    print(f"\n=== {nombre} ===")
    conteos[nombre], df_raros = revisar_dias(df)
    lista_raros.append(df_raros)

raros = pd.concat(lista_raros, ignore_index=True)
print(raros.shape)


=== Totalizadores Planta 2020_2022.xlsx ===
Cantidad de días según filas por día:
8        1
23       2
24       5
25     596
50       5
51       1
100      2
Name: count, dtype: int64

=== Totalizadores Planta 2021_2023.xlsx ===
Cantidad de días según filas por día:
9        1
16       1
23       3
24       3
25     586
50       5
51       1
99       1
100      1
Name: count, dtype: int64

=== Totalizadores Planta 2022_2023.xlsx ===
Cantidad de días según filas por día:
8       1
16      1
17      1
20      1
23      1
25    463
50      4
51      1
99      1
Name: count, dtype: int64
(1583, 21)


Hay como 100 días pegados. Saco ahora todos los valores que no sean 
- 23:59:00
- 23:59:01
- 23:59:59

In [6]:
def filtrar_2359(df):
    """Deja solo las filas con hora 23:59:xx (23:59:00, 23:59:01, 23:59:59)."""
    seg = pd.to_numeric(df.iloc[:, 1].map(a_segundos), errors="coerce")
    inicio = 23 * 3600 + 59 * 60          # 23:59:00
    fin = inicio + 59                     # 23:59:59
    return df[(seg >= inicio) & (seg <= fin)].copy()

In [7]:
dfs_2359 = {}
col_dia = "DIA"

for nombre, df in dfs.items():
    f = filtrar_2359(df)
    dfs_2359[nombre] = f

# Veo para una tabla:
f.info()

<class 'pandas.DataFrame'>
Index: 479 entries, 24 to 12000
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   DIA                   479 non-null    datetime64[us]
 1   HORA                  479 non-null    str           
 2   Planta (Kw)           479 non-null    float64       
 3   Elaboracion (Kw)      479 non-null    int64         
 4   Bodega (Kw)           479 non-null    int64         
 5   Cocina (Kw)           479 non-null    int64         
 6   Envasado (Kw)         479 non-null    int64         
 7   Linea 2 (Kw)          479 non-null    float64       
 8   Linea 3 (Kw)          479 non-null    int64         
 9   Linea 4 (Kw)          479 non-null    int64         
 10  Servicios (Kw)        479 non-null    int64         
 11  Sala Maq (Kw)         479 non-null    int64         
 12  Aire (Kw)             479 non-null    int64         
 13  Calderas (Kw)         479 non-nul

In [8]:
# Hago el dict con todo el df: 
# dfs_todo[archivo][hoja]
dfs_todo = {}

for archivo in archivos:
    print("Leyendo", archivo.name)
    dfs_todo[archivo.name] = pd.read_excel(archivo, sheet_name=None, engine="openpyxl")
    # sheet_name=None lee todas las hojas y devuelve un dict {nombre_hoja: df}

# Faltan hojas a un archivo
hojas_por_archivo = {n: set(h.keys()) for n, h in dfs_todo.items()}
todas = set.union(*hojas_por_archivo.values())
for n, h in hojas_por_archivo.items():
    print(n, len(h), "hojas | faltan:", todas - h)

Leyendo Totalizadores Planta 2020_2022.xlsx
Leyendo Totalizadores Planta 2021_2023.xlsx
Leyendo Totalizadores Planta 2022_2023.xlsx
Totalizadores Planta 2020_2022.xlsx 15 hojas | faltan: {'Seguimiento Dia'}
Totalizadores Planta 2021_2023.xlsx 16 hojas | faltan: set()
Totalizadores Planta 2022_2023.xlsx 15 hojas | faltan: {'Seguimiento Dia'}


Borro seguimiento día por que está en solo un excel

In [9]:
dfs_todo["Totalizadores Planta 2021_2023.xlsx"].pop("Seguimiento Dia")
dfs_todo["Totalizadores Planta 2021_2023.xlsx"].pop("Auxiliar")
dfs_todo["Totalizadores Planta 2020_2022.xlsx"].pop("Auxiliar")
dfs_todo["Totalizadores Planta 2022_2023.xlsx"].pop("Auxiliar")

,Unnamed: 0,DIA,HORA,Unnamed: 3,,.1,.2,.3,.4,.5,...,.21,.22,.23,.24,.25,.26,.27,.28,.29,.30
0,Ultima Fila sin Datos,2022-07-01,00:00:00,NaN,449270,449270.000000,1,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12012,2022-07-01,01:00:00,NaN,449270,449270.041667,0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Ultima Fila con Datos,2022-07-01,02:00:00,NaN,449270,449270.083333,0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12010,2022-07-01,03:00:00,NaN,449270,449270.125000,0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Diferencia,2022-07-01,04:00:01,NaN,449270,449270.166678,0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12004,NaN,2023-10-26,03:00:00,NaN,454080,454080.125000,0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12005,NaN,2023-10-26,04:00:00,NaN,454080,454080.166667,0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12006,NaN,2023-10-26,05:00:00,NaN,454080,454080.208333,0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12007,NaN,2023-10-26,06:00:00,NaN,454080,454080.250000,0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Union de los datos

In [10]:
unidos = {}
todas_las_hojas = sorted({h for hojas in dfs_todo.values() for h in hojas})

for hoja in todas_las_hojas:
    partes = []
    for nombre, hojas in dfs_todo.items():
        partes.append(filtrar_2359(hojas[hoja]))

    df = pd.concat(partes, ignore_index=True)
    df["DIA"] = pd.to_datetime(df["DIA"], errors="coerce").dt.date
    unidos[hoja] = df

    print(f"{hoja:35s} {df.shape} | DIA no interpretable: {df['DIA'].isna().sum()}")

Consolidado Agua                    (1712, 24) | DIA no interpretable: 0
Consolidado Aire                    (1712, 14) | DIA no interpretable: 0
Consolidado EE                      (1712, 21) | DIA no interpretable: 0
Consolidado GasVapor                (1712, 21) | DIA no interpretable: 0
Consolidado KPI                     (1712, 126) | DIA no interpretable: 0
Consolidado Produccion              (1712, 18) | DIA no interpretable: 0
Totalizadores Agua                  (1712, 44) | DIA no interpretable: 0
Totalizadores Aire                  (1712, 12) | DIA no interpretable: 0
Totalizadores CO2                   (1713, 9) | DIA no interpretable: 0
Totalizadores Efluentes             (1712, 9) | DIA no interpretable: 0
Totalizadores Energia               (1712, 60) | DIA no interpretable: 0
Totalizadores Gas y Vapor           (1712, 25) | DIA no interpretable: 0
Totalizadores Glicol                (1711, 12) | DIA no interpretable: 0
Totalizadores Produccion            (1712, 42) | DIA

In [11]:
with pd.ExcelWriter(
    ROOT / "data" / "processed" / "datos_base.xlsx",
    engine="openpyxl",
    date_format="YYYY/MM/DD", # Si no lo escribo así pone otra vez el 00:00:00
) as writer:
    for hoja, df in unidos.items():
        df.to_excel(writer, sheet_name=hoja[:31], index=False)

## Analisis del DF completo

In [12]:
ruta = ROOT / "data" / "processed" / "datos_base.xlsx"

unidos = pd.read_excel(ruta, sheet_name=None, engine="openpyxl")   # dict {hoja: df}

for hoja, df in unidos.items():
    print(f"{hoja:35s} {df.shape}")

Consolidado Agua                    (1712, 24)
Consolidado Aire                    (1712, 14)
Consolidado EE                      (1712, 21)
Consolidado GasVapor                (1712, 21)
Consolidado KPI                     (1712, 126)
Consolidado Produccion              (1712, 18)
Totalizadores Agua                  (1712, 44)
Totalizadores Aire                  (1712, 12)
Totalizadores CO2                   (1713, 9)
Totalizadores Efluentes             (1712, 9)
Totalizadores Energia               (1712, 60)
Totalizadores Gas y Vapor           (1712, 25)
Totalizadores Glicol                (1711, 12)
Totalizadores Produccion            (1712, 42)


Tienen sentido la cantidad de datos con días. Voy a juntar todo:

In [14]:
partes = []

for hoja, df in unidos.items():
    df = df.copy()
    df["DIA"] = pd.to_datetime(df["DIA"])
    df = df.sort_values("DIA", kind="stable")            # orden por fecha
    df["_n"] = df.groupby("DIA").cumcount()              # numera las filas de cada día para alinear entre hojas
    partes.append(df.set_index(["DIA", "_n"]))

df_total = pd.concat(partes, axis=1).sort_index()        # todas las columnas, tal cual vienen
df_total = df_total.reset_index(level="_n", drop=True)   # queda DIA como índice

print(df_total.shape)
df_total.head()

(1714, 423)


,HORA,Agua Planta (Hl),Agua Elaboracion (Hl),Agua Bodega (Hl),Agua Cocina (Hl),Agua Dilucion (Hl),Agua Envasado (Hl),Agua Servicios (Hl),Planta de agua (Hl),Produccion (Hl),...,Hl Cerveza L2,Hl Cerveza L3,Hl Cerveza L4,Hl Cerveza L5,Cocimientos Diarios,Nivel Silo Bagazo Norte (1),Nivel Silo Bagazo Sur (2),Nivel Tk Restos Lev,Hl Session IPA,Id
DIA,,,,,,,,,,,,,,,,,,,,,
2020-07-01,23:59:00,10280.0,1306.0,1015.0,291.0,0.0,1092.0,7882.0,11241.40,2398.0,...,74.0,0.0,12.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2020-07-02,23:59:00,13970.0,15464.3,9933.3,5531.0,3874.0,9684.2,-11178.5,22107.77,25148.5,...,713.0,4753.0,5201.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2020-07-03,23:59:00,36300.0,17008.7,8598.7,8410.0,3074.0,12356.3,6935.0,46955.43,29365.0,...,33.0,4575.0,5416.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2020-07-04,23:59:00,40120.0,19224.2,7377.3,11846.9,2251.0,12926.6,7969.2,51124.18,32150.8,...,1348.0,5250.0,5314.0,311.0,NaN,NaN,NaN,NaN,NaN,NaN
2020-07-05,23:59:00,38940.0,18716.2,9265.0,9451.2,4446.0,12077.7,8146.1,49146.08,30793.9,...,1103.0,1380.0,5498.0,377.0,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
df_total.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 1714 entries, 2020-07-01 to 2023-10-25
Columns: 423 entries, HORA to Id
dtypes: float64(404), object(5), str(14)
memory usage: 5.5+ MB


Muy importante, hay 5 objetcs y 14 strings, hay que buscarlas y corregirlas seguramente. Primero igual tengo la sospecha que consolidado KPI tiene las columnas del resto de los valores:

In [21]:
import numpy as np
import pandas as pd
from collections import defaultdict

# Si df_total tiene columnas MultiIndex (hoja, columna), las aplano a solo el nombre
if isinstance(df_total.columns, pd.MultiIndex):
    df_total.columns = df_total.columns.get_level_values(-1)

# Solo numéricas y con variación (las constantes dan falsos positivos)
X = df_total.select_dtypes("number")
X = X.loc[:, (X.nunique() > 1).values]
X = X.loc[:, ~X.columns.astype(str).str.startswith("Unnamed")]

# Etiqueta única por columna: "nombre [posición]"
X.columns = [f"{c} [{i}]" for i, c in enumerate(X.columns)]
print(X.shape)

(1714, 357)


In [22]:
base = pd.Series(X.columns).str.replace(r"\s*\[\d+\]$", "", regex=True).str.lower().str.strip()
rep = base[base.duplicated(keep=False)]
print(rep.groupby(rep).apply(lambda s: list(X.columns[s.index])).to_string())

cocimientos diarios        [Cocimientos Diarios [189], Cocimientos Diario...
fc barriles                            [FC Barriles [15], FC Barriles [198]]
fc l1 y l2                               [FC L1 y L2 [10], FC L1 y L2 [192]]
fc lavadora l2                   [FC Lavadora L2 [11], FC Lavadora L2 [217]]
fc lavadora l3                   [FC Lavadora L3 [13], FC Lavadora L3 [215]]
hl cerveza l2                     [Hl Cerveza L2 [185], Hl Cerveza L2 [348]]
hl cerveza l3                     [Hl Cerveza L3 [186], Hl Cerveza L3 [349]]
hl cerveza l4                     [Hl Cerveza L4 [187], Hl Cerveza L4 [350]]
hl cerveza l5                     [Hl Cerveza L5 [188], Hl Cerveza L5 [351]]
id                         [Id [229], Id [239], Id [246], Id [248], Id [2...
kw gral planta                   [KW Gral Planta [52], KW Gral Planta [249]]
red l1 y l2                             [Red L1 y L2 [9], Red L1 y L2 [191]]
red l3                                           [Red L3 [12], Red L3 [194]]

In [23]:
grupos = defaultdict(list)
for col in X.columns:
    h = pd.util.hash_pandas_object(X[col].round(4), index=False).sum()
    grupos[h].append(col)

identicas = [cols for cols in grupos.values() if len(cols) > 1]
for cols in identicas:
    print(cols)
print(f"\n{len(identicas)} grupos de columnas idénticas")

['Red L1 y L2 [9]', 'Red L1 y L2 [191]']
['FC L1 y L2 [10]', 'FC L1 y L2 [192]']
['FC Lavadora L2 [11]', 'FC Lavadora L2 [217]']
['Red L3 [12]', 'Red L3 [194]']
['FC Lavadora L3 [13]', 'FC Lavadora L3 [215]']
['Red Paste L4 [14]', 'Red Paste L4 [196]']
['FC Barriles [15]', 'FC Barriles [198]']
['Aire Producido (M3) [22]', 'Totalizador Sistema Kaeser [236]']
['Totalizador_Aire_Bodega [28]', 'Totalizador_Aire_Bodega [234]']
['Totalizador_Aire_Cocina [29]', 'Totalizador_Aire_Cocina [235]']
['Totalizador_Aire_L2 [30]', 'Totalizador_Aire_L2 [230]']
['Totalizador_Aire_L3 [31]', 'Totalizador_Aire_L3 [231]']
['Totaliador_Aire_L4 [32]', 'Totaliador_Aire_L4 [232]']
['Totalizador_Aire_L5 [33]', 'Totalizador_Aire_L5 [233]']
['KW Gral Planta [52]', 'KW Gral Planta [249]']
['Vapor Servicio (Kg) [58]', 'Tot_Vapor_CIP_Bodega [66]', 'Tot_Vapor_CIP_Bodega [301]']
['Tot_Vapor_L3_L4 [62]', 'Tot_Vapor_L3_L4 [300]']
['VAPOR DE LINEA 1 Y 2 KG [63]', 'VAPOR DE LINEA 1 Y 2 KG [306]']
['VAPOR DE LINEA 4 KG [64]

In [24]:
corr = X.corr()
mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)
pares = corr.where(mask).stack().reset_index()
pares.columns = ["a", "b", "r"]

umbral = 0.999                         # bajalo a 0.98 para ver las "demasiado parecidas"
pares = pares[pares["r"].abs() > umbral].sort_values("r", ascending=False)
print(len(pares), "pares con |r| >", umbral)
print(pares.to_string(index=False))

581 pares con |r| > 0.999
                            a                                b         r
                     Id [323]                         Id [356]  1.000000
         Agua Cocina (Hl) [3]                  FC Cocina [202]  1.000000
              Red L1 y L2 [9]                Red L1 y L2 [191]  1.000000
              FC L1 y L2 [10]                 FC L1 y L2 [192]  1.000000
          FC Lavadora L2 [11]             FC Lavadora L2 [217]  1.000000
                     Id [239]                         Id [323]  1.000000
                     Id [239]                         Id [317]  1.000000
                     Id [239]                         Id [298]  1.000000
                     Id [239]                         Id [248]  1.000000
                     Id [229]                         Id [356]  1.000000
                     Id [229]                         Id [323]  1.000000
                     Id [229]                         Id [317]  1.000000
                     Id [

La verdad que c,laude hizo esto. Me voy a dormir y mañana lo hago/interpreto/pienso/reniego.